In [ ]:
#!/usr/bin/env python3
"""
Volleyball Stats CLI
A command-line interface for managing and analyzing volleyball match data.
"""

import csv
import json
import os
import sys
from collections import defaultdict
from datetime import datetime

# ============================================================================
# DATA LOADING
# ============================================================================

def load_unique_teams():
    teams_by_id = {}
    teams_by_name = defaultdict(set)
    if not os.path.exists("unique_teams.csv"):
        return teams_by_id, teams_by_name
    with open("unique_teams.csv", 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            teams_by_id[row['Team_ID']] = row['Team_Name']
            teams_by_name[row['Team_Name']].add(row['Team_ID'])
    return teams_by_id, teams_by_name

def load_all_matches():
    matches = []
    if not os.path.exists("all_matches.csv"):
        return matches
    with open("all_matches.csv", 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            matches.append(row)
    return matches

# ============================================================================
# COMMANDS
# ============================================================================

def cmd_extract():
    print("\n" + "=" * 50)
    print("EXTRACTING UNIQUE TEAMS")
    print("=" * 50)
    
    teams_by_id = {}
    teams_by_name = defaultdict(set)
    conflicts = []  # Track naming conflicts
    
    if not os.path.exists("Data"):
        print("❌ No Data folder found.")
        return teams_by_id, teams_by_name
    
    csv_files = [f for f in os.listdir("Data") if f.endswith('.csv')]
    
    if not csv_files:
        print("❌ No CSV files found in Data folder.")
        return teams_by_id, teams_by_name
    
    print(f"Found {len(csv_files)} event files\n")
    
    for filename in csv_files:
        filepath = os.path.join("Data", filename)
        print(f"  Processing: {filename}")
        
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                # Team A
                if row.get('Team_A_ID') and row.get('Team_A_Name'):
                    existing_name = teams_by_id.get(row['Team_A_ID'])
                    if existing_name and existing_name != row['Team_A_Name']:
                        conflicts.append((row['Team_A_ID'], existing_name, row['Team_A_Name']))
                    teams_by_id[row['Team_A_ID']] = row['Team_A_Name']
                    teams_by_name[row['Team_A_Name']].add(row['Team_A_ID'])
                
                # Team B
                if row.get('Team_B_ID') and row.get('Team_B_Name'):
                    existing_name = teams_by_id.get(row['Team_B_ID'])
                    if existing_name and existing_name != row['Team_B_Name']:
                        conflicts.append((row['Team_B_ID'], existing_name, row['Team_B_Name']))
                    teams_by_id[row['Team_B_ID']] = row['Team_B_Name']
                    teams_by_name[row['Team_B_Name']].add(row['Team_B_ID'])
    
    # Save unique teams
    with open("unique_teams.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Team_ID', 'Team_Name'])
        for name in sorted(teams_by_name.keys()):
            for team_id in sorted(teams_by_name[name]):
                writer.writerow([team_id, name])
    
    # Save ID mapping
    with open("team_id_mapping.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Team_ID', 'Team_Name'])
        for team_id in sorted(teams_by_id.keys(), key=lambda x: int(x) if x.isdigit() else 0):
            writer.writerow([team_id, teams_by_id[team_id]])
    
    print(f"\n✅ Extracted {len(teams_by_id)} unique teams")
    print("   Saved: unique_teams.csv, team_id_mapping.csv")
    
    # Report conflicts
    if conflicts:
        print(f"\n⚠️  Found {len(conflicts)} naming conflicts (same ID, different names):")
        for tid, old_name, new_name in conflicts[:5]:
            print(f"   ID {tid}: '{old_name}' → '{new_name}'")
        if len(conflicts) > 5:
            print(f"   ... and {len(conflicts) - 5} more")
    else:
        print("   ✅ No naming conflicts found")
    
    return teams_by_id, teams_by_name

def cmd_merge():
    print("\n" + "=" * 50)
    print("MERGING ALL EVENTS")
    print("=" * 50)
    
    all_rows = []
    headers = None
    seen_match_ids = set()
    
    if not os.path.exists("Data"):
        print("❌ No Data folder found.")
        return
    
    event_files = [f for f in os.listdir("Data") if f.endswith('.csv')]
    
    if not event_files:
        print("❌ No event files found.")
        return
    
    print(f"Merging {len(event_files)} files...\n")
    
    for filename in event_files:
        filepath = os.path.join("Data", filename)
        event_id = filename.replace('.csv', '')
        
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            if headers is None:
                headers = ['Event_ID'] + reader.fieldnames
            
            for row in reader:
                match_id = row.get('Match_ID', '')
                
                # Skip duplicates
                if match_id and match_id in seen_match_ids:
                    print(f"  ⏭ Skipped duplicate: {match_id}")
                    continue
                
                if match_id:
                    seen_match_ids.add(match_id)
                
                row['Event_ID'] = event_id
                all_rows.append(row)
        
        print(f"  Added: {filename}")
    
    with open("all_matches.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(all_rows)
    
    duplicates_removed = len(seen_match_ids) - len(all_rows) if len(seen_match_ids) > len(all_rows) else 0
    
    print(f"\n✅ Merged {len(all_rows)} unique matches into 'all_matches.csv'")
    if duplicates_removed > 0:
        print(f"   (Removed {duplicates_removed} duplicate matches)")
    else:
        print("   ✅ No duplicate matches found")

def cmd_search(query):
    print("\n" + "=" * 50)
    print(f"SEARCH: {query}")
    print("=" * 50)
    
    teams_by_id, teams_by_name = load_unique_teams()
    
    if not teams_by_name:
        print("❌ No teams found. Run 'extract' first.")
        return
    
    results = [(name, ids) for name, ids in teams_by_name.items() 
               if query.lower() in name.lower()]
    
    if not results:
        print(f"No teams found matching '{query}'")
        return
    
    print(f"Found {len(results)} result(s):\n")
    print(f"{'#':<4} {'Team Name':<40} {'Team IDs':<20}")
    print("-" * 70)
    
    for i, (name, ids) in enumerate(sorted(results, key=lambda x: x[0]), 1):
        ids_str = ', '.join(sorted(ids))
        print(f"{i:<4} {name:<40} {ids_str:<20}")

def cmd_view(team_name):
    print("\n" + "=" * 50)
    print(f"VIEW: {team_name}")
    print("=" * 50)
    
    teams_by_id, teams_by_name = load_unique_teams()
    matches = load_all_matches()
    
    if not teams_by_name:
        print("❌ No teams found. Run 'extract' and 'merge' first.")
        return
    
    team_ids = []
    matched_names = []
    for name, ids in teams_by_name.items():
        if team_name.lower() in name.lower():
            team_ids.extend(ids)
            matched_names.append(name)
    
    if not team_ids:
        print(f"No team found matching '{team_name}'")
        return
    
    print(f"\nFound {len(matched_names)} team(s):")
    for name in matched_names:
        ids = teams_by_name[name]
        print(f"  • {name}: {', '.join(sorted(ids))}")
    
    if not matches:
        print("\n❌ No matches file found. Run 'merge' first.")
        return
    
    print("\n--- Match History ---")
    
    match_count = 0
    wins = 0
    losses = 0
    
    for row in matches:
        if row['Team_A_ID'] in team_ids or row['Team_B_ID'] in team_ids:
            match_count += 1
            
            is_team_a = row['Team_A_ID'] in team_ids
            opp_name = row['Team_B_Name'] if is_team_a else row['Team_A_Name']
            won = (is_team_a and row['Team_A_Won_Match'] == 'True') or \
                  (not is_team_a and row['Team_B_Won_Match'] == 'True')
            
            if won:
                wins += 1
                result = "✅ W"
            else:
                losses += 1
                result = "❌ L"
            
            scores = []
            for i in range(1, 4):
                s_a = row.get(f'Set{i}_TeamA', '')
                s_b = row.get(f'Set{i}_TeamB', '')
                if s_a and s_b:
                    scores.append(f"{s_a}-{s_b}")
            
            score_str = ', '.join(scores) if scores else "N/A"
            
            print(f"  [{result}] vs {opp_name}")
            print(f"       {score_str} | {row['Division']} | {row['Time']}")
    
    if match_count > 0:
        print(f"\n--- Summary ---")
        print(f"Total: {match_count} matches ({wins}W - {losses}L)")
    else:
        print("\n⚠ No matches found for this team.")

def cmd_compare(team1, team2):
    print("\n" + "=" * 50)
    print(f"COMPARE: {team1} vs {team2}")
    print("=" * 50)
    
    teams_by_id, teams_by_name = load_unique_teams()
    matches = load_all_matches()
    
    if not teams_by_name:
        print("❌ No teams found. Run 'extract' and 'merge' first.")
        return
    
    team1_ids = []
    team1_name = None
    for name, ids in teams_by_name.items():
        if team1.lower() in name.lower():
            team1_ids.extend(ids)
            if not team1_name:
                team1_name = name
    
    team2_ids = []
    team2_name = None
    for name, ids in teams_by_name.items():
        if team2.lower() in name.lower():
            team2_ids.extend(ids)
            if not team2_name:
                team2_name = name
    
    if not team1_ids:
        print(f"❌ Team 1 not found: '{team1}'")
        return
    if not team2_ids:
        print(f"❌ Team 2 not found: '{team2}'")
        return
    
    print(f"\n{team1_name} ({', '.join(sorted(team1_ids))})")
    print(f"vs")
    print(f"{team2_name} ({', '.join(sorted(team2_ids))})")
    
    if not matches:
        print("\n❌ No matches file. Run 'merge' first.")
        return
    
    h2h_matches = []
    for row in matches:
        t1_in_a = row['Team_A_ID'] in team1_ids
        t1_in_b = row['Team_B_ID'] in team1_ids
        t2_in_a = row['Team_A_ID'] in team2_ids
        t2_in_b = row['Team_B_ID'] in team2_ids
        
        if (t1_in_a and t2_in_b) or (t1_in_b and t2_in_a):
            h2h_matches.append(row)
    
    if not h2h_matches:
        print("\n⚠ No head-to-head matches found.")
        return
    
    t1_wins = 0
    t2_wins = 0
    
    print(f"\n--- Head-to-Head ({len(h2h_matches)} matches) ---")
    print(f"{'Result':<8} {'Score':<20} {'Division':<15} {'Date'}")
    print("-" * 60)
    
    for row in h2h_matches:
        is_t1_a = row['Team_A_ID'] in team1_ids
        
        if is_t1_a:
            won = row['Team_A_Won_Match'] == 'True'
        else:
            won = row['Team_B_Won_Match'] == 'True'
        
        if won:
            t1_wins += 1
            result = f"{team1_name[:15]} W"
        else:
            t2_wins += 1
            result = f"{team1_name[:15]} L"
        
        scores = []
        for i in range(1, 4):
            s_a = row.get(f'Set{i}_TeamA', '')
            s_b = row.get(f'Set{i}_TeamB', '')
            if s_a and s_b:
                scores.append(f"{s_a}-{s_b}")
        score_str = ', '.join(scores)
        
        date = row['Time'][:10] if row.get('Time') else 'N/A'
        
        print(f"{result:<8} {score_str:<20} {row['Division']:<15} {date}")
    
    print("-" * 60)
    print(f"SERIES: {team1_name} {t1_wins} - {t2_wins} {team2_name}")

def cmd_list():
    print("\n" + "=" * 50)
    print("ALL TEAMS")
    print("=" * 50)
    
    teams_by_id, teams_by_name = load_unique_teams()
    
    if not teams_by_name:
        print("❌ No teams found. Run 'extract' first.")
        return
    
    print(f"\nTotal: {len(teams_by_name)} unique team names, {len(teams_by_id)} IDs\n")
    print(f"{'#':<4} {'Team Name':<40} {'IDs'}")
    print("-" * 60)
    
    for i, (name, ids) in enumerate(sorted(teams_by_name.items(), key=lambda x: x[0]), 1):
        ids_str = ', '.join(sorted(ids))
        print(f"{i:<4} {name:<40} {ids_str}")

def cmd_stats():
    print("\n" + "=" * 50)
    print("STATISTICS")
    print("=" * 50)
    
    teams_by_id, teams_by_name = load_unique_teams()
    matches = load_all_matches()
    
    print("\n📊 Data Overview:")
    print(f"   Unique team names: {len(teams_by_name)}")
    print(f"   Unique team IDs:   {len(teams_by_id)}")
    print(f"   Total matches:     {len(matches)}")
    
    event_ids = set()
    if matches:
        event_ids = set(row.get('Event_ID', '') for row in matches if row.get('Event_ID'))
    print(f"   Events loaded:      {len(event_ids)}")
    
    team_matches = defaultdict(int)
    for row in matches:
        if row.get('Team_A_ID'):
            team_matches[row['Team_A_ID']] += 1
        if row.get('Team_B_ID'):
            team_matches[row['Team_B_ID']] += 1
    
    if team_matches:
        top_teams = sorted(team_matches.items(), key=lambda x: x[1], reverse=True)[:10]
        print("\n🏆 Most Active Teams:")
        for tid, count in top_teams:
            name = teams_by_id.get(tid, 'Unknown')
            print(f"   {name:<30} {count} matches")
    
    print("\n📁 Files:")
    for fname in ['unique_teams.csv', 'all_matches.csv', 'team_id_mapping.csv']:
        exists = "✅" if os.path.exists(fname) else "❌"
        print(f"   {exists} {fname}")

def cmd_export(fmt='csv'):
    print("\n" + "=" * 50)
    print(f"EXPORT: {fmt.upper()}")
    print("=" * 50)
    
    matches = load_all_matches()
    teams_by_id, teams_by_name = load_unique_teams()
    
    if not matches:
        print("❌ No matches found. Run 'extract' and 'merge' first.")
        return
    
    fmt = fmt.lower()
    
    if fmt == 'csv':
        print("Data already available as 'all_matches.csv'")
        
    elif fmt == 'json':
        output_file = "all_matches.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(matches, f, indent=2, ensure_ascii=False)
        print(f"✅ Exported to '{output_file}'")
        
    elif fmt == 'html':
        output_file = "all_matches.html"
        html = """<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Volleyball Match Report</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1 { color: #333; }
        table { border-collapse: collapse; width: 100%; margin-top: 20px; }
        th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
        th { background-color: #4CAF50; color: white; }
        tr:nth-child(even) { background-color: #f2f2f2; }
        .wins { color: green; font-weight: bold; }
        .loss { color: red; }
        .stats { background: #f9f9f9; padding: 15px; margin: 20px 0; border-radius: 5px; }
    </style>
</head>
<body>
    <h1>🏐 Volleyball Match Report</h1>
    <div class="stats">
        <strong>Total Matches:</strong> """ + str(len(matches)) + """<br>
        <strong>Total Teams:</strong> """ + str(len(teams_by_id)) + """
    </div>
    <table>
        <tr><th>Date</th><th>Division</th><th>Team A</th><th>Team B</th><th>Score</th><th>Court</th></tr>
"""
        for row in matches[:1000]:
            date = row.get('Time', '')[:10]
            div = row.get('Division', '')
            team_a = row.get('Team_A_Name', '')
            team_b = row.get('Team_B_Name', '')
            court = row.get('Court', '')
            
            scores = []
            for i in range(1, 4):
                s_a = row.get(f'Set{i}_TeamA', '')
                s_b = row.get(f'Set{i}_TeamB', '')
                if s_a and s_b:
                    scores.append(f"{s_a}-{s_b}")
            score = ', '.join(scores) if scores else '-'
            
            won = row.get('Team_A_Won_Match', 'False') == 'True'
            result_class = "wins" if won else "loss"
            
            html += f"<tr><td>{date}</td><td>{div}</td><td class=\"{result_class}\">{team_a}</td><td>{team_b}</td><td>{score}</td><td>{court}</td></tr>\n"
        
        html += """
    </table>
</body>
</html>
"""
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(html)
        print(f"✅ Exported to '{output_file}'")
        
    elif fmt == 'teams':
        output_file = "teams_list.json"
        teams_list = [{"id": tid, "name": name} for tid, name in teams_by_id.items()]
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(teams_list, f, indent=2, ensure_ascii=False)
        print(f"✅ Exported {len(teams_list)} teams to '{output_file}'")
        
    else:
        print(f"❌ Unknown format: '{fmt}'")
        print("Available formats: csv, json, html, teams")

def cmd_ranking(event_id=None):
    print("\n" + "=" * 50)
    print("TOURNAMENT RANKINGS")
    print("=" * 50)
    
    matches = load_all_matches()
    teams_by_id, teams_by_name = load_unique_teams()
    
    if not matches:
        print("❌ No matches found. Run 'merge' first.")
        return
    
    if event_id:
        filtered_matches = [m for m in matches if event_id.lower() in m.get('Event_ID', '').lower()]
        if not filtered_matches:
            print(f"❌ No matches found for event: '{event_id}'")
            return
        print(f"\nFiltering by: {event_id}\n")
    else:
        filtered_matches = matches
    
    team_stats = defaultdict(lambda: {'wins': 0, 'losses': 0, 'sets_won': 0, 'sets_lost': 0, 'points_for': 0, 'points_against': 0})
    
    for row in filtered_matches:
        t_a_id = row.get('Team_A_ID', '')
        t_b_id = row.get('Team_B_ID', '')
        
        if not t_a_id or not t_b_id:
            continue
        
        a_won = row.get('Team_A_Won_Match', 'False') == 'True'
        
        for i in range(1, 4):
            s_a = row.get(f'Set{i}_TeamA', '')
            s_b = row.get(f'Set{i}_TeamB', '')
            if s_a and s_b:
                try:
                    sa, sb = int(s_a), int(s_b)
                    team_stats[t_a_id]['points_for'] += sa
                    team_stats[t_a_id]['points_against'] += sb
                    team_stats[t_b_id]['points_for'] += sb
                    team_stats[t_b_id]['points_against'] += sa
                    if sa > sb:
                        team_stats[t_a_id]['sets_won'] += 1
                        team_stats[t_b_id]['sets_lost'] += 1
                    else:
                        team_stats[t_b_id]['sets_won'] += 1
                        team_stats[t_a_id]['sets_lost'] += 1
                except:
                    pass
        
        if a_won:
            team_stats[t_a_id]['wins'] += 1
            team_stats[t_b_id]['losses'] += 1
        else:
            team_stats[t_b_id]['wins'] += 1
            team_stats[t_a_id]['losses'] += 1
    
    rankings = []
    for tid, stats in team_stats.items():
        total = stats['wins'] + stats['losses']
        if total > 0:
            win_pct = stats['wins'] / total
            point_diff = stats['points_for'] - stats['points_against']
            rankings.append({
                'id': tid,
                'name': teams_by_id.get(tid, 'Unknown'),
                'wins': stats['wins'],
                'losses': stats['losses'],
                'win_pct': win_pct,
                'sets_won': stats['sets_won'],
                'sets_lost': stats['sets_lost'],
                'point_diff': point_diff
            })
    
    rankings.sort(key=lambda x: (-x['win_pct'], -x['point_diff']))
    
    print(f"{'Rank':<5} {'Team':<30} {'W-L':<8} {'Win%':<8} {'Sets':<10} {'+/−'}")
    print("-" * 75)
    
    for i, r in enumerate(rankings, 1):
        sets = f"{r['sets_won']}-{r['sets_lost']}"
        win_pct = f"{r['win_pct']:.1%}"
        pd = f"+{r['point_diff']}" if r['point_diff'] >= 0 else str(r['point_diff'])
        print(f"{i:<5} {r['name']:<30} {r['wins']}-{r['losses']:<5} {win_pct:<8} {sets:<10} {pd}")
    
    print(f"\n📊 {len(rankings)} teams ranked")

def cmd_filter(date_from=None, date_to=None):
    print("\n" + "=" * 50)
    print("FILTER MATCHES BY DATE")
    print("=" * 50)
    
    matches = load_all_matches()
    
    if not matches:
        print("❌ No matches found. Run 'merge' first.")
        return
    
    if not date_from and not date_to:
        print("Usage: filter <YYYY-MM-DD> [YYYY-MM-DD]")
        print("Example: filter 2025-01-01 2025-12-31")
        return
    
    try:
        if date_from:
            date_from_obj = datetime.strptime(date_from, "%Y-%m-%d")
        else:
            date_from_obj = datetime.min
        
        if date_to:
            date_to_obj = datetime.strptime(date_to, "%Y-%m-%d")
        else:
            date_to_obj = datetime.max
        
    except ValueError:
        print("❌ Invalid date format. Use YYYY-MM-DD")
        return
    
    filtered = []
    for row in matches:
        time_str = row.get('Time', '')
        if time_str:
            try:
                match_date = datetime.strptime(time_str[:10], "%Y-%m-%d")
                if date_from_obj <= match_date <= date_to_obj:
                    filtered.append(row)
            except:
                pass
    
    if not filtered:
        print(f"No matches found between {date_from} and {date_to}")
        return
    
    print(f"\nFound {len(filtered)} matches:\n")
    print(f"{'Date':<12} {'Teams':<50} {'Score'}")
    print("-" * 80)
    
    for row in filtered[:50]:
        date = row.get('Time', '')[:10]
        matchup = f"{row.get('Team_A_Name', '')} vs {row.get('Team_B_Name', '')}"
        
        scores = []
        for i in range(1, 4):
            s_a = row.get(f'Set{i}_TeamA', '')
            s_b = row.get(f'Set{i}_TeamB', '')
            if s_a and s_b:
                scores.append(f"{s_a}-{s_b}")
        score = ', '.join(scores)
        
        print(f"{date:<12} {matchup:<50} {score}")
    
    if len(filtered) > 50:
        print(f"\n... and {len(filtered) - 50} more matches")

def cmd_dedupe():
    """Remove duplicates from all CSV files in Data folder."""
    print("\n" + "=" * 50)
    print("DEDUPLICATING FILES")
    print("=" * 50)
    
    if not os.path.exists("Data"):
        print("❌ No Data folder found.")
        return
    
    csv_files = [f for f in os.listdir("Data") if f.endswith('.csv')]
    
    if not csv_files:
        print("❌ No CSV files found in Data folder.")
        return
    
    print(f"Found {len(csv_files)} event files\n")
    
    total_removed = 0
    
    for filename in csv_files:
        filepath = os.path.join("Data", filename)
        seen_ids = set()
        unique_rows = []
        headers = None
        duplicates = 0
        
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            headers = reader.fieldnames
            for row in reader:
                match_id = row.get('Match_ID', '')
                if match_id and match_id in seen_ids:
                    duplicates += 1
                    continue
                if match_id:
                    seen_ids.add(match_id)
                unique_rows.append(row)
        
        # Write deduplicated file
        with open(filepath, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            writer.writeheader()
            writer.writerows(unique_rows)
        
        print(f"  {filename}: {len(unique_rows)} unique, {duplicates} removed")
        total_removed += duplicates
    
    print(f"\n✅ Removed {total_removed} duplicate matches across all files")

def cmd_help():
    print("\n" + "=" * 50)
    print("🏐 VOLLEYBALL STATS CLI - HELP")
    print("=" * 50)
    print("""
COMMANDS:
  extract              Extract unique teams from event CSVs
  merge                Merge all events into all_matches.csv (deduped)
  search <name>        Search for teams by name
  view <name>          View team details and match history
  compare <t1> vs <t2> Compare two teams head-to-head
  list                 List all teams
  stats                Show overall statistics
  export <format>      Export data (csv, json, html, teams)
  ranking [event]      Show team rankings (optionally by event)
  filter <date_from> [date_to]  Filter matches by date range
  dedupe               Remove duplicates from individual event files
  help                 Show this help
  quit                 Exit the program

EXAMPLES:
  > extract
  > merge
  > dedupe
  > list
  > stats
  > search "St. James"
  > view "Elite Volleyball"
  > compare "St. James" vs "Elite"
  > ranking
  > ranking PTAwMDAwNDEyODc90
  > filter 2025-01-01 2025-12-31
  > export json
  > export html
  > quit
""")

# ============================================================================
# MAIN CLI LOOP
# ============================================================================

def main():
    print("\n" + "=" * 50)
    print("🏐 VOLLEYBALL STATS CLI")
    print("=" * 50)
    print("Type 'help' for commands.\n")
    
    while True:
        try:
            user_input = input("📌 > ").strip()
            
            if not user_input:
                continue
            
            parts = user_input.split()
            cmd = parts[0].lower()
            args = parts[1:]
            
            if cmd in ('quit', 'exit', 'q'):
                print("Goodbye! 👋")
                break
            
            elif cmd == 'help':
                cmd_help()
            
            elif cmd == 'extract':
                cmd_extract()
            
            elif cmd == 'merge':
                cmd_merge()
            
            elif cmd == 'dedupe':
                cmd_dedupe()
            
            elif cmd == 'search':
                if not args:
                    print("Usage: search <team_name>")
                else:
                    cmd_search(' '.join(args))
            
            elif cmd == 'view':
                if not args:
                    print("Usage: view <team_name>")
                else:
                    cmd_view(' '.join(args))
            
            elif cmd == 'compare':
                try:
                    sep_idx = next((i for i, a in enumerate(args) if a.lower() in ('vs', 'v', 'and', '&', ',')), -1)
                    if sep_idx > 0:
                        team1 = ' '.join(args[:sep_idx])
                        team2 = ' '.join(args[sep_idx+1:])
                    elif len(args) >= 2:
                        team1 = args[0]
                        team2 = ' '.join(args[1:])
                    else:
                        print("Usage: compare <team1> vs <team2>")
                        continue
                    cmd_compare(team1, team2)
                except Exception as e:
                    print(f"Error: {e}")
            
            elif cmd == 'list':
                cmd_list()
            
            elif cmd == 'stats':
                cmd_stats()
            
            elif cmd == 'export':
                fmt = args[0] if args else 'csv'
                cmd_export(fmt)
            
            elif cmd == 'ranking':
                event_id = args[0] if args else None
                cmd_ranking(event_id)
            
            elif cmd == 'filter':
                date_from = args[0] if len(args) > 0 else None
                date_to = args[1] if len(args) > 1 else None
                cmd_filter(date_from, date_to)
            
            else:
                print(f"Unknown command: '{cmd}'. Type 'help' for commands.")
        
        except KeyboardInterrupt:
            print("\nUse 'quit' to exit.")
        except Exception as e:
            print(f"Error: {e}")

if __name__ == "__main__":
    main()
